# MS3: EDA, Baseline Modeling, and Pipeline Development

**Canvas Project #150**

**Project:** Hyperbolic Embeddings for Hierarchical Style Representation

**Group Members:** Peter Flo, Luca Grossmann, Valerie Wang

**COMPSCI 2090B — Spring 2026**

## 1. Problem Statement Refinement and Introduction

Develop an introduction to your project (can be a brief summary of your Milestone 2)

Reiterate/refine your problem statement if needed. Ensure it is clear and understandable to someone outside your field.

Articulate significance based on new insights from EDA, highlighting shifts in perspective or understanding.

## 2. Comprehensive EDA Review:
Include key findings from your extended EDA, focusing on how these insights informed feature engineering and model selection.

Detail the feature engineering process, including transformations, encodings, or selection techniques, with justifications rooted in EDA.

Include visualizations that support analytical decisions, linking back to the problem statement and project goals.

## 3. Baseline Model Selection and Justification:
Explain the choice of your baseline model, considering simplicity, interpretability, and relevance with respect to your project statement.

Define the data used for training and possibly testing. Be sure to address whether entire data set was considered, and if not, why.

Detail the training process, including preprocessing, parameter choices and/or tuning, and metrics used for evaluation.

### Baseline Model

### Data Preprocessing

To preporcess the data for our baseline model, we had to prepare the data for the CLIP ViT-B/16 image encoder which we are going to keep frozen during training.

To do this, we:

1. **Resize** the shorter side to 224.
2. **CenterCrop** to 224×224.
3. **Convert to RGB** 
4. **ToTensor** — `PIL` → `float32` tensor `(3, 224, 224)`.
5. **Normalize** with CLIP's mean/std:
   - mean = `(0.48145466, 0.4578275, 0.40821073)`
   - std  = `(0.26862954, 0.26130258, 0.27577711)`

In addition to this, we set up a 70/30 train/test/val split. We have additional data processing work that we will need to include in our workflow (class balancing, class masking, data augmentation). However, for the baseline we decided it would be best to start with a fairly raw dataset. 



### Embedding

To avoid passing the data through Clip on every forward pass, we did one pass through of our entire dataset and saved the output. 

The outputted data is much lower dimensional, and has (some) emeded semantic feauteres of the dataset. 

``` Python
    model, _, preprocess = open_clip.create_model_and_transforms(
        "ViT-B-16", pretrained="openai", device=device)
    model.eval()

    loader = DataLoader(WikiArtDataset(paths, preprocess),
                        batch_size=128, num_workers=8, shuffle=False)

    chunks = []
    t0 = time.time()
    with torch.no_grad():
        for batch in tqdm(loader):
            chunks.append(model.encode_image(batch.to(device)).cpu().numpy())
    features = np.concatenate(chunks).astype(np.float16)

``` 

To run this code and get the embeddings locally:

`python extract_clip_features.py`

### Training Process

#### Preprocessing 

We start our training process by loading in our embedding matrix `clip_vitb16.npy` and the labels `index.csv`. 



**Parameter Choices**
- For initial hparam choices, we used Claude Opus recomnedations. So far we have done minimal hparam tuning, but plan to set up a grid search due to how fast the model trains. 

**Loss Functions**


**Metrics used for Eval**: TODO

#### Initial Results

Present initial results, discussing alignment with expectations and project objectives.

## 4. Results Interpretation and Analysis:
Analyze the baseline model’s performance, using appropriate metrics and visualizations.

Assess strengths and weaknesses of the model, proposing improvements for the next iteration.

Propose future directions: what methods, models, and adjustments do you expect to incorporate for the final project.  These should be grounded based on insights gathered from the data, baseline models, and/or outside resources.

### Euclidean vs. Hyperbolic Validation Results

We evaluated both models on the **24,421-image validation split** using the Section 2 evaluation plan: tree distortion, dendrogram agreement, kNN retrieval consistency, style-classification accuracy, and Frechet-mean interpolation stability. Both models use frozen CLIP ViT-B/16 image features and an 8-dimensional learned embedding head; the Euclidean model is the baseline, while the hyperbolic model uses a Poincare-ball embedding and Riemannian prototype optimization.

#### How These Results Were Produced

- Training entry points: `scripts/train.py`
- Evaluation entry point: `scripts/eval.py`
- Validation metrics used below came from:
  - `data/runs/euclidean_d8_e10/eval_val_section2/metrics.json`
  - `data/runs/hyperbolic_d8_e10/eval_val_section2/metrics.json`
- Supporting diagnostics came from:
  - `tree_distortion_pairs.csv`
  - `dendrogram_clusters.csv`
  - `frechet_interpolation_trials.csv`

To reproduce these runs locally:

```bash
python - <<'PY'
from pathlib import Path
from scripts.train import train_euclidean, train_hyperbolic

train_euclidean(dim=8, epochs=10, batch_size=4096, run_dir=Path('data/runs/euclidean_d8_e10'), device='cpu')
train_hyperbolic(dim=8, epochs=10, batch_size=4096, run_dir=Path('data/runs/hyperbolic_d8_e10'), device='cpu')
PY

python scripts/eval.py --ckpt data/runs/euclidean_d8_e10/ckpt.pt --split val --batch-size 4096 --device cpu --output-dir data/runs/euclidean_d8_e10/eval_val_section2
python scripts/eval.py --ckpt data/runs/hyperbolic_d8_e10/ckpt.pt --split val --batch-size 4096 --device cpu --output-dir data/runs/hyperbolic_d8_e10/eval_val_section2
```

Note: `data/` is gitignored, so the raw plots and CSV outputs are generated locally rather than stored in the repository.

#### Metric Summary

| Metric | Euclidean d=8 | Hyperbolic d=8 | Better |
|---|---:|---:|---|
| Top-1 accuracy | 54.3% | 47.3% | Euclidean |
| Top-5 accuracy | 92.4% | 89.8% | Euclidean |
| Balanced accuracy | 37.2% | 30.0% | Euclidean |
| Class-center/tree Spearman | 0.292 | 0.189 | Euclidean |
| Average tree distortion | 1.782 | 1.856 | Euclidean |
| Worst-case tree distortion | 5.339 | 6.599 | Euclidean |
| Dendrogram cluster F1 | 0.051 | 0.000 | Euclidean |
| Sibling recall@5 | 0.199 | 0.219 | Hyperbolic |
| Sibling recall@10 | 0.281 | 0.328 | Hyperbolic |
| Cousin recall@5 | 0.287 | 0.315 | Hyperbolic |
| Cousin recall@10 | 0.365 | 0.405 | Hyperbolic |
| Cubism Frechet-mean nearest-prototype accuracy | 0.91 | 0.39 | Euclidean |

#### Main Takeaways

- **Euclidean wins most global metrics.** It outperforms hyperbolic on top-1/top-5 accuracy, balanced accuracy, centroid-to-tree correlation, tree distortion, dendrogram agreement, and interpolation stability.
- **Hyperbolic wins the local retrieval metrics.** Sibling and cousin recall are both higher for the hyperbolic embedding at `k=5` and `k=10`.
- **This is a meaningful negative result.** At `d=8` and 10 epochs, hyperbolic geometry does not yet improve global hierarchy recovery over the Euclidean baseline.

#### Interpretation

The Euclidean baseline is currently the stronger overall model. It reaches **54.3% top-1 accuracy** compared with **47.3%** for hyperbolic, and its balanced accuracy is also noticeably higher (**37.2%** vs. **30.0%**). That matters because balanced accuracy reduces the influence of large classes like Impressionism and Realism, so the Euclidean advantage is not just coming from majority-class dominance.

The hierarchy-aware metrics point in the same direction. Euclidean achieves a higher centroid/tree Spearman correlation (**0.292** vs. **0.189**), lower average tree distortion (**1.782** vs. **1.856**), lower worst-case distortion (**5.339** vs. **6.599**), and a small but nonzero dendrogram cluster F1 (**0.051** vs. **0.000**). In other words, under the current objective and training budget, Euclidean is recovering more of the **global** art-historical structure.

Hyperbolic does show one encouraging pattern: it performs better on **local subtree retrieval**. Sibling recall and cousin recall are both higher at `k=5` and `k=10`, which suggests that hyperbolic geometry may already be organizing **nearby branches** more effectively even though it is not yet winning on global structure or classification. That makes the result more nuanced than a simple failure.

The Frechet-mean interpolation experiment also currently favors Euclidean. A mean over 8 sampled Cubism embeddings lands nearest to the Cubism prototype in **91%** of Euclidean trials but only **39%** of hyperbolic trials, and the hyperbolic run has a negative average margin to the nearest competing prototype. For now, Euclidean averaging is much more stable.

#### What This Means For The Project

At this stage, the results do **not** support the claim that hyperbolic geometry is already outperforming Euclidean geometry on the main research question. That is still a useful finding. It suggests that frozen CLIP features already encode a large amount of style structure in a form that a simple Euclidean head can exploit, and that switching to hyperbolic geometry alone is not enough.

The most natural next steps are:

- Sweep embedding dimension and curvature instead of testing only `d=8, c=1`.
- Train the hyperbolic model longer and tune its optimizer separately rather than reusing the Euclidean budget.
- Add a more explicitly hierarchy-aware objective instead of relying only on prototype classification.
- Keep using the same evaluation harness so every comparison stays apples-to-apples.

Local confusion matrices and diagnostic CSVs were generated under `data/runs/.../eval_val_section2/`, but those files are not tracked in git because `data/` is gitignored.


## 5. Final Model Pipeline Setup:
Outline steps for a pipeline for your final models, describing each component from data preprocessing to evaluation.

The pipeline can be laid out as a diagram, as an algorithmic outline, or with code or pseudocode.,,s

Document assumptions, parameter choices, and preliminary tuning considerations.